# LSA64 preprocessing → 128×128 frames

Decodes all 3200 LSA64 clips once, centre-cropped and resized to 128×128,
so training is GPU-bound rather than CPU-bound (Kaggle gives ~4 cores, which
cannot feed two T4s from raw video).

CPU-only: **costs no GPU quota**. Measured at ~0.70 s/clip → ~37 min.

Per D16 this kernel does **not** authenticate or publish. It writes to
`/kaggle/working` and stops; the workstation fetches the output and pushes it
as a dataset. Nothing here depends on a credential.

Uses `justinvo277/lsa64-dataset`, the **cut** distribution (D14) — the paper
does not say which it used, and cut removes idle frames that would inflate
L1 and TCD for free.

In [ ]:
import subprocess, sys

r = subprocess.run(
    ["git", "clone", "-q", "--branch", "feat/m0-m1-harness", "--depth", "1",
     "https://github.com/SonLamHG/pgmm.git", "/kaggle/working/repo"],
    capture_output=True, text=True,
)
assert r.returncode == 0, r.stderr
sys.path.insert(0, "/kaggle/working/repo")
print("repo cloned")

In [ ]:
# Discover the source clips. The mount layout is /kaggle/input/datasets/<owner>/
# <slug>/... -- not the commonly documented /kaggle/input/<slug> -- so glob for
# it rather than trusting either shape.
from pathlib import Path

INPUT = Path("/kaggle/input")
mp4s = sorted(INPUT.rglob("*.mp4"))
assert len(mp4s) == 3200, f"expected 3200 clips, found {len(mp4s)}"
SRC = mp4s[0].parent.parent  # .../LSA64  (clips are grouped in per-sign dirs)
print("clips :", len(mp4s))
print("source:", SRC)

In [ ]:
# Preprocess all 3200 clips through the tested entry point. prepare_dataset()
# recurses, so it handles this distribution's per-sign nesting, and writes a
# flat clip-id-keyed output plus index.json.
import time

from pgmm.data.lsa64_prepare import prepare_dataset

OUT = Path("/kaggle/working/lsa64_prepared")

t0 = time.time()
index = prepare_dataset(SRC, OUT, size=128)
dt = time.time() - t0

print(f"done: {len(index)} clips, {sum(index.values())} frames in {dt/60:.1f} min")
print(f"      {dt/len(index):.2f}s/clip (smoke test projected 0.70)")

In [ ]:
# Verify the output, and measure its size -- that size IS the round-trip cost
# that local orchestration (D16) pays, so it decides whether that choice holds.
import cv2

assert len(index) == 3200, f"expected 3200 prepared clips, got {len(index)}"
assert all(n > 0 for n in index.values()), "some clip decoded to zero frames"

frames = sorted(OUT.rglob("frame_*.jpg"))
total_bytes = sum(f.stat().st_size for f in frames)
print("clips        :", len(index))
print("frames       :", len(frames))
print(f"total size   : {total_bytes/2**30:.2f} GiB")
print(f"mean frame   : {total_bytes/len(frames)/1024:.1f} KiB")
print(f"frames/clip  : min {min(index.values())}, max {max(index.values())}, "
      f"mean {sum(index.values())/len(index):.1f}")

img = cv2.imread(str(frames[0]))
assert img.shape == (128, 128, 3), img.shape
print("frame shape  :", img.shape, "OK")

In [ ]:
# The prepared frames must survive as the kernel's output. Kaggle keeps
# /kaggle/working, but the cloned repo is also in there and would be published
# alongside the data -- remove it so the output is exactly the dataset.
import shutil

shutil.rmtree("/kaggle/working/repo", ignore_errors=True)
leftovers = [p.name for p in Path("/kaggle/working").iterdir()]
print("kernel output will contain:", leftovers)